In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import pipeline, AutoConfig
from sklearn.metrics import accuracy_score, classification_report

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    pipeline_device = torch.device('mps')
else:
    device = 'cpu'
    pipeline_device = -1

print({'selected_device': device})


In [ ]:
dataset = load_dataset('dair-ai/emotion', split='test')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

print({'dataset': 'dair-ai/emotion', 'split': 'test', 'num_rows': len(dataset)})
print(dataset[:3])


In [ ]:
model_name = 'j-hartmann/emotion-english-distilroberta-base'
config = AutoConfig.from_pretrained(model_name)

clf = pipeline(
    task='text-classification',
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True
)

id2label = {}
if hasattr(config, 'id2label') and config.id2label is not None:
    for k, v in config.id2label.items():
        id2label[int(k)] = str(v)

label2id = {}
if hasattr(config, 'label2id') and config.label2id is not None:
    label2id = {str(k): int(v) for k, v in config.label2id.items()}

print({'model_name': model_name, 'id2label': id2label, 'label2id': label2id})


In [ ]:
canonical_set = set(class_names)
alias_map = {
    'sadness': 'sadness',
    'sad': 'sadness',
    'joy': 'joy',
    'happy': 'joy',
    'happiness': 'joy',
    'love': 'love',
    'anger': 'anger',
    'angry': 'anger',
    'fear': 'fear',
    'fearful': 'fear',
    'scared': 'fear',
    'surprise': 'surprise',
    'surprised': 'surprise'
}

def normalize_label(raw_label):
    label = str(raw_label).strip()
    upper_label = label.upper()
    if upper_label.startswith('LABEL_'):
        idx = int(label.split('_')[-1])
        label = id2label.get(idx, label)
    label = str(label).strip().lower()
    label = alias_map.get(label, label)
    if label not in canonical_set:
        raise ValueError(f'Unrecognized label: {raw_label} -> {label}')
    return label

label_to_id = {name: i for i, name in enumerate(class_names)}
print({'class_names': class_names, 'label_to_id': label_to_id})


In [ ]:
texts = dataset['text']
true_ids = dataset['label']
batch_size = 32

pred_outputs = clf(texts, batch_size=batch_size, top_k=2)

pred_labels = []
pred_scores = []
top2_labels = []
top2_scores = []
confidence_margins = []

for item in pred_outputs:
    if not isinstance(item, list):
        item = [item]
    if len(item) < 2:
        item = item + [{'label': item[0]['label'], 'score': 0.0}]
    first = item[0]
    second = item[1]
    first_label = normalize_label(first['label'])
    second_label = normalize_label(second['label'])
    first_score = float(first['score'])
    second_score = float(second['score'])
    pred_labels.append(first_label)
    pred_scores.append(first_score)
    top2_labels.append(second_label)
    top2_scores.append(second_score)
    confidence_margins.append(first_score - second_score)

pred_ids = [label_to_id[label] for label in pred_labels]

results_df = pd.DataFrame({
    'text': texts,
    'true_label': [class_names[i] for i in true_ids],
    'predicted_label_top1': pred_labels,
    'score_top1': pred_scores,
    'predicted_label_top2': top2_labels,
    'score_top2': top2_scores,
    'confidence_margin_top1_minus_top2': confidence_margins,
    'correct_top1': [int(t == p) for t, p in zip(true_ids, pred_ids)]
})

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'test',
    'num_examples': len(dataset),
    'device': device,
    'accuracy': round(float(accuracy), 6),
    'mean_top1_confidence': round(float(np.mean(pred_scores)), 6),
    'mean_top2_confidence': round(float(np.mean(top2_scores)), 6),
    'mean_confidence_margin': round(float(np.mean(confidence_margins)), 6),
    'median_confidence_margin': round(float(np.median(confidence_margins)), 6)
})
print(report)


In [ ]:
sample_n = 8
print(results_df.head(sample_n).to_string(index=False))
